In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

print("Libraries Loaded")

In [ ]:
#data loading
load_dotenv()
DATA_PATH_2025 = os.getenv("LOL_DATA_PATH_2025")
DATA_PATH_2026 = os.getenv("LOL_DATA_PATH_2026")


df_2025 = pd.read_csv(DATA_PATH_2025)
df_2026 = pd.read_csv(DATA_PATH_2026)
df = df = pd.concat([df_2025, df_2026], ignore_index=True)

lck = df[(df['position'] == 'team') & (df['league'] == 'LCK')]
print(lck.groupby(['year', 'split', 'teamname'])['gameid'].count().reset_index())


In [ ]:
#data processing
lck['date'] = pd.to_datetime(lck['date'])
lck = lck.sort_values('date').reset_index(drop=True)

cols = ['gameid', 'date', 'patch', "side", 'teamname', 'result', 
        'golddiffat15', 'firstdragon', 'firstherald', 
        'firsttower']

lck = lck[cols]

print(lck.shape)
print(lck.head())

In [ ]:
lck = lck.sort_values(['teamname', 'date']).reset_index(drop=True)

def add_rolling_features(df, window=5):
    df = df.copy()
    
    for team, group in df.groupby('teamname'):
        idx = group.index

        df.loc[idx, 'roll_winrate'] = (
            group['result'].shift(1).rolling(window, min_periods=1).mean()
        )

        df.loc[idx, 'roll_golddiff15'] = (
            group['golddiffat15'].shift(1).rolling(window, min_periods=1).mean()
        )
        
        df.loc[idx, 'roll_firstdragon'] = (
            group['firstdragon'].shift(1).rolling(window, min_periods=1).mean()
        )

        df.loc[idx, 'roll_firstherald'] = (
            group['firstherald'].shift(1).rolling(window, min_periods=1).mean()
        )

        df.loc[idx, 'roll_firsttower'] = (
            group['firsttower'].shift(1).rolling(window, min_periods=1).mean()
        )
        
        df.loc[idx, 'patch_winrate'] = (
            group.groupby('patch')['result']
            .transform(lambda x: x.shift(1).expanding().mean())
        )
    
    return df

lck_featured = add_rolling_features(lck, window=5)
print(lck_featured[['teamname', 'date', 'patch', 'result', 
           'roll_winrate', 'roll_golddiff15', 'patch_winrate']].head(10))
print(lck_featured.head(5))

In [ ]:
print(lck_featured[lck_featured['gameid'] == 'LOLTMNT03_183532'][['gameid', 'teamname', 'side']])

In [ ]:
blue = lck_featured[lck_featured['side'] == 'Blue'].copy()
red = lck_featured[lck_featured['side'] == 'Red'].copy()

feature_cols = ['roll_winrate', 'roll_golddiff15', 'roll_firstdragon', 
                'roll_firstherald', 'roll_firsttower', 'patch_winrate']

blue = blue.rename(columns={col: f'blue_{col}' for col in feature_cols})
red = red.rename(columns={col: f'red_{col}' for col in feature_cols})

merged = pd.merge(
    blue[['gameid', 'date', 'patch', 'result'] + [f'blue_{col}' for col in feature_cols]],
    red[['gameid'] + [f'red_{col}' for col in feature_cols]],
    on='gameid'
)

for col in feature_cols:
    merged[f'diff_{col}'] = merged[f'blue_{col}'] - merged[f'red_{col}']

print(merged.shape)
print(merged[['gameid', 'date', 'result'] + [f'diff_{col}' for col in feature_cols]].head())

In [ ]:
print(merged.isna().sum())
print(f"\ncol: {len(merged)}")
print(f"col with NaN: {merged.isna().any(axis=1).sum()}")


In [ ]:
diff_cols = [f'diff_{col}' for col in feature_cols]

model_df = merged.copy()
model_df['diff_patch_winrate'] = model_df['diff_patch_winrate'].fillna(0)

model_df['blue_patch_winrate'] = model_df['blue_patch_winrate'].fillna(0.5)
model_df['red_patch_winrate'] = model_df['red_patch_winrate'].fillna(0.5)

model_df['diff_patch_winrate'] = model_df['blue_patch_winrate'] - model_df['red_patch_winrate']

model_df = model_df.dropna(subset=[c for c in diff_cols if c != 'diff_patch_winrate']).copy()
model_df = model_df.sort_values('date').reset_index(drop=True)

print(f"Entire mathces: {len(merged)} matches")
print(f"training data: {len(model_df)}th matches")
print(f"dropped matches: {len(merged) - len(model_df)} matches")
print(f"\nlef Na:")
print(model_df[diff_cols].isna().sum())

In [ ]:
import warnings
warnings.filterwarnings('ignore')

X = model_df[diff_cols]
y = model_df['result']

tscv = TimeSeriesSplit(n_splits=5)

models = {
    'Logistic Regression': LogisticRegression(),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

for name, model in models.items():
    accs, aucs = [], []
    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_test_sc = scaler.transform(X_test)
        
        model.fit(X_train_sc, y_train)
        preds = model.predict(X_test_sc)
        proba = model.predict_proba(X_test_sc)[:, 1]
        
        accs.append(accuracy_score(y_test, preds))
        aucs.append(roc_auc_score(y_test, proba))
    
    print(f"{name}")
    print(f"  Accuracy: {np.mean(accs):.3f} (+/- {np.std(accs):.3f})")
    print(f"  ROC-AUC:  {np.mean(aucs):.3f} (+/- {np.std(aucs):.3f})")

In [ ]:
scaler_final = StandardScaler()
X_scaled = scaler_final.fit_transform(X)

lr_final = LogisticRegression()
lr_final.fit(X_scaled, y)

coef = pd.Series(lr_final.coef_[0], index=diff_cols).sort_values()

label_map = {
    'diff_roll_winrate':     'Recent Win Rate',
    'diff_roll_golddiff15':  'Gold Diff @15',
    'diff_roll_firstdragon': 'First Dragon',
    'diff_roll_firstherald': 'First Herald',
    'diff_roll_firsttower':  'First Tower',
    'diff_patch_winrate':    'Patch Win Rate'
}
coef.index = [label_map[i] for i in coef.index]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#d73027' if v < 0 else '#1a9850' for v in coef.values]
bars = ax.barh(coef.index, coef.values, color=colors, edgecolor='white', height=0.6)

ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Coefficient (Blue - Red)', fontsize=11)
ax.set_title('Feature Importance — LCK 2025 Win Prediction\n(Logistic Regression)', fontsize=13, fontweight='bold')
ax.set_xlim(min(coef.values) * 1.3, max(coef.values) * 1.3)

for bar, val in zip(bars, coef.values):
    ax.text(val + (0.003 if val >= 0 else -0.003),
            bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}',
            va='center', ha='left' if val >= 0 else 'right', fontsize=10)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("saved: feature_importance.png")

In [ ]:
import joblib

joblib.dump(lr_final, 'lck_model.pkl')
joblib.dump(scaler_final, 'lck_scaler.pkl')

def get_team_features(team_name, df=lck_featured):
    team_df = df[df['teamname'] == team_name].sort_values('date')
    latest = team_df.iloc[-1]
    return {
        'roll_winrate':     latest['roll_winrate'],
        'roll_golddiff15':  latest['roll_golddiff15'],
        'roll_firstdragon': latest['roll_firstdragon'],
        'roll_firstherald': latest['roll_firstherald'],
        'roll_firsttower':  latest['roll_firsttower'],
        'patch_winrate':    latest['patch_winrate'] if pd.notna(latest['patch_winrate']) else 0.5
    }

def predict_match(blue_team, red_team):
    blue = get_team_features(blue_team)
    red = get_team_features(red_team)
    
    diff = {f'diff_{k}': blue[k] - red[k] for k in blue.keys()}
    
    X_pred = pd.DataFrame([diff])[diff_cols]
    X_pred_sc = scaler_final.transform(X_pred)
    
    proba = lr_final.predict_proba(X_pred_sc)[0]
    
    print(f"\n{blue_team} (Blue) vs {red_team} (Red)")
    print(f"  {blue_team} winrate: {proba[1]:.1%}")
    print(f"  {red_team} winrate:  {proba[0]:.1%}")
    print(f"  predicted winner: {blue_team if proba[1] > 0.5 else red_team}")
    
    return proba

predict_match('T1', 'Hanwha Life Esports')
predict_match('KT Rolster', 'Dplus Kia')

In [ ]:
os.makedirs('../backend/models', exist_ok=True)
os.makedirs('../backend/data', exist_ok=True)

joblib.dump(lr_final, '../backend/models/lck_model.pkl')
joblib.dump(scaler_final, '../backend/models/lck_scaler.pkl')
joblib.dump(lck_featured, '../backend/data/lck_featured.pkl')

print("saved pkl")